In [ ]:
import dlt
from pyspark.sql.functions import (
    col, when, sum, count, avg, max, min, coalesce, lit, current_date, current_timestamp,
    year, month, dayofweek, hour, date_format, round, greatest
)
from pyspark.sql.types import DecimalType, IntegerType, DateType, StringType, DoubleType


In [ ]:
# ===============================================================================
#                     SIMPLIFIED GOLD LAYER CONFIGURATION
# ===============================================================================
env = spark.conf.get("pipeline.env") 
catalog = "raiffka_catalog"
# Bronze schema is always "bronze" - no environment prefix
bronze_schema = "bronze"
# Silver and Gold schemas use environment prefix
silver_schema = f"{env}_silver" if env else "silver"
gold_schema = f"{env}_gold" if env else "gold"
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {gold_schema}")

# Business constants
CURRENT_DATE = current_date()
CURRENT_TIMESTAMP = current_timestamp()


In [ ]:
#------------------------------- SIMPLIFIED TRANSACTION ANALYTICS -------------------------
@dlt.table(
    name="fact_transaction_analytics_simplified",
    comment="Simplified transaction analytics for basic business insights - DLT streaming compatible",
    table_properties={
        "quality": "gold",
        "delta.autoOptimize.optimizeWrite": "true",
        "delta.autoOptimize.autoCompact": "true"
    },
    partition_cols=["transaction_year", "transaction_month"]
)
def fact_transaction_analytics_simplified():
    # Use simple transaction tables instead of complex enriched fact
    card_trans = dlt.readStream(f"{catalog}.{silver_schema}.card_transactions_silver")
    financial_trans = dlt.readStream(f"{catalog}.{silver_schema}.financial_transactions_silver")
    customers = dlt.readStream(f"{catalog}.{silver_schema}.dim_customers")
    accounts = dlt.readStream(f"{catalog}.{silver_schema}.dim_accounts")
    cards = dlt.readStream(f"{catalog}.{silver_schema}.dim_cards")
    cities = dlt.read(f"{catalog}.{silver_schema}.cities_silver")
    regions = dlt.read(f"{catalog}.{silver_schema}.regions_silver")
    
    # Simple card transactions with customer info
    card_with_customer = (
        card_trans.alias("ct")
        .join(cards.filter(col("__END_AT").isNull()).alias("card"), 
              col("ct.card_id") == col("card.card_id"), "inner")
        .join(accounts.filter(col("__END_AT").isNull()).alias("acc"), 
              col("card.account_id") == col("acc.account_id"), "inner")
        .join(customers.filter(col("__END_AT").isNull()).alias("cust"), 
              col("acc.customer_id") == col("cust.customer_id"), "inner")
        .join(cities.alias("city"), col("ct.city_id") == col("city.city_id"), "left")
        .join(regions.alias("region"), col("ct.region_id") == col("region.region_id"), "left")
        .select(
            col("ct.card_transaction_id").alias("transaction_id"),
            lit("CARD").alias("transaction_type"),
            col("ct.card_id"),
            col("acc.account_id"),
            col("cust.customer_id"),
            col("ct.merchant_id"),
            col("ct.amount"),
            col("ct.datum").alias("transaction_date"),
            col("ct.typ_transakce").alias("merchant_category"),
            # Geography info
            coalesce(col("city.nazev"), lit("Unknown")).alias("city_name"),
            coalesce(col("region.nazev"), lit("Unknown")).alias("region_name"),
            # Business metrics
            when(col("ct.amount") > 1000, "High Value")
            .when(col("ct.amount") > 100, "Medium Value")
            .otherwise("Low Value").alias("transaction_value_segment"),
            hour(col("ct.datum")).alias("transaction_hour"),
            dayofweek(col("ct.datum")).alias("transaction_day_of_week"),
            month(col("ct.datum")).alias("transaction_month"),
            year(col("ct.datum")).alias("transaction_year"),
            # Customer context
            col("cust.client_age"),
            col("cust.prijem").alias("customer_income")
        )
    )
    
    # Simple financial transactions with customer info
    financial_with_customer = (
        financial_trans.alias("ft")
        .join(accounts.filter(col("__END_AT").isNull()).alias("acc"), 
              col("ft.account_id") == col("acc.account_id"), "inner")
        .join(customers.filter(col("__END_AT").isNull()).alias("cust"), 
              col("acc.customer_id") == col("cust.customer_id"), "inner")
        .join(cities.alias("city"), col("ft.city_id") == col("city.city_id"), "left")
        .join(regions.alias("region"), col("ft.region_id") == col("region.region_id"), "left")
        .select(
            col("ft.transaction_id"),
            lit("FINANCIAL").alias("transaction_type"),
            lit(None).cast("string").alias("card_id"),
            col("acc.account_id"),
            col("cust.customer_id"),
            lit(None).cast("string").alias("merchant_id"),
            col("ft.amount"),
            col("ft.datum").alias("transaction_date"),
            col("ft.typ_transakce").alias("merchant_category"),
            # Geography info
            coalesce(col("city.nazev"), lit("Unknown")).alias("city_name"),
            coalesce(col("region.nazev"), lit("Unknown")).alias("region_name"),
            # Business metrics
            when(col("ft.amount") > 1000, "High Value")
            .when(col("ft.amount") > 100, "Medium Value")
            .otherwise("Low Value").alias("transaction_value_segment"),
            hour(col("ft.datum")).alias("transaction_hour"),
            dayofweek(col("ft.datum")).alias("transaction_day_of_week"),
            month(col("ft.datum")).alias("transaction_month"),
            year(col("ft.datum")).alias("transaction_year"),
            # Customer context
            col("cust.client_age"),
            col("cust.prijem").alias("customer_income")
        )
    )
    
    # Union both transaction types
    return card_with_customer.union(financial_with_customer)


In [ ]:
#------------------------------- BASIC BUSINESS METRICS TABLE -------------------------
@dlt.table(
    name="business_metrics_enhanced",
    comment="Enhanced business metrics table - DLT streaming compatible",
    table_properties={
        "quality": "gold",
        "delta.autoOptimize.optimizeWrite": "true",
        "delta.autoOptimize.autoCompact": "true"
    }
)
def business_metrics_enhanced():
    transactions = dlt.readStream(f"{catalog}.{gold_schema}.fact_transaction_analytics_simplified")
    
    return (
        transactions
        .select(
            # Transaction basics
            col("transaction_id"),
            col("transaction_type"),
            col("customer_id"),
            col("amount"),
            col("transaction_date"),
            col("transaction_value_segment"),
            col("city_name"),
            col("region_name"),
            
            # Simple derived metrics
            when(col("transaction_hour").between(9, 17), "Business Hours")
            .when(col("transaction_hour").between(18, 23), "Evening")
            .otherwise("Night").alias("time_period"),
            
            when(col("transaction_day_of_week").isin([1, 7]), "Weekend")
            .otherwise("Weekday").alias("day_type"),
            
            when(col("customer_income") > 50000, "High Income")
            .when(col("customer_income") > 25000, "Medium Income")
            .otherwise("Low Income").alias("income_segment"),
            
            # Timestamp
            CURRENT_TIMESTAMP.alias("processed_at")
        )
    )


In [ ]:
#------------------------------- CUSTOMER SUMMARY TABLE -------------------------
@dlt.table(
    name="customer_demographics_enhanced",
    comment="Enhanced customer demographics table - no complex aggregations",
    table_properties={
        "quality": "gold",
        "delta.autoOptimize.optimizeWrite": "true",
        "delta.autoOptimize.autoCompact": "true"
    }
)
def customer_demographics_enhanced():
    customers = dlt.readStream(f"{catalog}.{silver_schema}.dim_customers")
    
    return (
        customers
        .filter(col("__END_AT").isNull())  # Current customers only
        .select(
            col("customer_id"),
            col("jmeno"),
            col("prijmeni"),
            col("client_age"),
            col("prijem").alias("annual_income"),
            col("pohlavi").alias("gender"),
            col("pracovni_stav").alias("employment_status"),
            col("mesto").alias("city"),
            col("zeme").alias("country"),
            
            # Simple customer categorization
            when(col("client_age") < 25, "Young")
            .when(col("client_age") < 45, "Middle-aged")
            .when(col("client_age") < 65, "Mature")
            .otherwise("Senior").alias("age_group"),
            
            when(col("prijem") > 80000, "Premium")
            .when(col("prijem") > 50000, "Gold")
            .when(col("prijem") > 25000, "Silver")
            .otherwise("Standard").alias("customer_tier"),
            
            CURRENT_TIMESTAMP.alias("processed_at")
        )
    )


In [ ]:
#------------------------------- DAILY TRANSACTION SUMMARY -------------------------
@dlt.table(
    name="daily_transaction_summary",
    comment="Daily transaction summary without complex aggregations - DLT compatible",
    table_properties={
        "quality": "gold",
        "delta.autoOptimize.optimizeWrite": "true",
        "delta.autoOptimize.autoCompact": "true"
    }
)
def daily_transaction_summary():
    transactions = dlt.readStream(f"{catalog}.{gold_schema}.fact_transaction_analytics_simplified")
    
    return (
        transactions
        .select(
            # Date dimensions
            col("transaction_date"),
            col("transaction_year"),
            col("transaction_month"),
            date_format(col("transaction_date"), "EEEE").alias("day_name"),
            
            # Transaction details
            col("transaction_type"),
            col("transaction_value_segment"),
            col("amount"),
            
            # Geography
            col("city_name"),
            col("region_name"),
            
            # Customer context
            col("customer_id"),
            when(col("client_age") < 30, "Young")
            .when(col("client_age") < 50, "Middle")
            .otherwise("Senior").alias("customer_age_group"),
            
            when(col("customer_income") > 60000, "High Income")
            .when(col("customer_income") > 30000, "Medium Income")
            .otherwise("Standard Income").alias("customer_income_tier"),
            
            # Time patterns
            when(col("transaction_hour").between(6, 12), "Morning")
            .when(col("transaction_hour").between(12, 18), "Afternoon")
            .when(col("transaction_hour").between(18, 22), "Evening")
            .otherwise("Night").alias("time_of_day"),
            
            # Weekend indicator
            when(dayofweek(col("transaction_date")).isin([1, 7]), lit(True))
            .otherwise(lit(False)).alias("is_weekend"),
            
            # Merchant info (for card transactions)
            col("merchant_id"),
            col("merchant_category"),
            
            # Processing metadata
            CURRENT_TIMESTAMP.alias("processed_at")
        )
    )
